# Step 5 – linker_v5 results (AIDev v5)
Reads `results/` from `step4_run_linker_v5.py` and shows:
1. **Result 1** – % of AI PRs per agent with a follow-up (30 / 60 days, all_changes vs fix_only)
2. **What kind of change** is the follow-up (fix, feat, refactor…) → Q1
3. **Same agent or another agent? How fast?**
4. **Result 2** – the list of pairs (AI PR → follow-up PR, shared files) → Excel

> These are **candidate** follow-ups: same files + later date does **not** prove the AI PR was wrong. AIDev v5 has no human PRs, so only agent follow-ups are counted.

In [ ]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt
from paths import RAW_V5, RESULTS

BLUE, BRICK, MID, GREY = "#0032A0", "#BA6F4C", "#13155C", "#5A6B7D"   # GVSU colours
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
os.makedirs("tables", exist_ok=True); os.makedirs("charts", exist_ok=True)
AGENTS = {"Claude_Code": "Claude Code", "OpenAI_Codex": "Codex", "Copilot": "Copilot",
          "Devin": "Devin", "Cursor": "Cursor", "Google_Jules": "Jules"}
NET = os.path.exists(os.path.join(RESULTS, "net_all_changes_30_60", "per_agent.csv"))
pre = "net_" if NET else ""      # net diff (paper H3) when step4b was run
print("file overlap:", "NET DIFF (like the paper)" if NET else "commit files")
runs = {m: os.path.join(RESULTS, f"{pre}{m}_30_60") for m in ("all_changes", "fix_only")}
for m, p in runs.items():
    md = json.load(open(os.path.join(p, "run_metadata.json")))
    print(f"{m:12s}: {md['n_original_prs']:,} merged AI PRs | {md['n_candidate_pairs']:,} candidate pairs")

## 1. Result 1 – % of AI PRs with a follow-up, per agent

In [ ]:
rows = []
for m, p in runs.items():
    s = pd.read_csv(os.path.join(p, "per_agent.csv"))
    for _, r in s.iterrows():
        rows.append({"agent": AGENTS.get(r.agent, r.agent), "AI PRs": int(r.n_prs), "mode": m,
                     "30 days": round(r.pct_followup_30d_unc, 1), "60 days": round(r.pct_followup_60d_unc, 1)})
long = pd.DataFrame(rows)
per_agent = long.pivot_table(index=["agent", "AI PRs"], columns="mode", values=["30 days", "60 days"]).round(1)
per_agent.columns = [f"{m} – {w}" for w, m in per_agent.columns]
per_agent = per_agent[["all_changes – 30 days", "fix_only – 30 days", "all_changes – 60 days", "fix_only – 60 days"]]
per_agent = per_agent.sort_values("all_changes – 30 days", ascending=False).reset_index()
per_agent.to_csv("tables/1_per_agent.csv", index=False)
display(per_agent)
print("% over uncensored PRs (the whole 30 / 60-day window is inside the data)")

In [ ]:
order = per_agent.agent.tolist()
d = per_agent.set_index("agent").loc[order]
x = range(len(order)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 4.8))
b1 = ax.bar([i - w/2 for i in x], d["all_changes – 30 days"], w, color=BLUE, label="all_changes (any later change)")
b2 = ax.bar([i + w/2 for i in x], d["fix_only – 30 days"], w, color=BRICK, label="fix_only (later change called fix)")
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.8, f"{b.get_height():.1f}%", ha="center", fontsize=9)
ax.set_xticks(list(x)); ax.set_xticklabels([f"{a}\n(n={n:,})" for a, n in zip(order, d['AI PRs'])])
ax.set_ylabel("% of AI PRs"); ax.set_ylim(0, 75)
ax.set_title("AI PRs with at least one follow-up within 30 days (AIDev v5, Python)", fontsize=12)
ax.legend(frameon=False, loc="upper right")
fig.tight_layout(); fig.savefig("charts/1_followup_rate_per_agent.png", dpi=200); plt.show()

## 2. What kind of change is the follow-up? (all_changes, 60 days) → Q1

In [ ]:
pairs = pd.read_parquet(os.path.join(runs["all_changes"], "candidate_pairs.parquet"))
pairs["followup_task_type"] = pairs["followup_task_type"].fillna("(none)")
types = pairs.followup_task_type.value_counts()
pct = (types / types.sum() * 100).round(1)
display(pd.DataFrame({"pairs": types, "%": pct}))
fig, ax = plt.subplots(figsize=(8, 4.2))
cols = [BRICK if t == "fix" else BLUE for t in pct.index]
ax.barh(pct.index[::-1], pct.values[::-1], color=cols[::-1])
for i, v in enumerate(pct.values[::-1]):
    ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlabel("% of follow-up pairs"); ax.set_title(f"Type of the follow-up PR ({types.sum():,} pairs, all_changes)", fontsize=12)
fig.tight_layout(); fig.savefig("charts/2_followup_types.png", dpi=200); plt.show()

## 3. Same agent or another agent? How fast?

In [ ]:
pairs["who"] = (pairs.original_agent == pairs.followup_agent).map({True: "same agent", False: "other agent"})
who = pd.crosstab(pairs.original_agent.map(AGENTS), pairs.who, normalize="index").mul(100).round(1)
who["pairs"] = pairs.original_agent.map(AGENTS).value_counts()
display(who.sort_values("pairs", ascending=False))
print(f"Overall: {(pairs.who == 'same agent').mean():.1%} of follow-ups are by the same agent")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.hist(pairs.days_after_merge, bins=60, color=BLUE)
ax.axvline(30, color=BRICK, ls="--"); ax.text(30.5, ax.get_ylim()[1] * 0.9, "30 days", color=BRICK)
ax.set_xlabel("days after the AI PR was merged"); ax.set_ylabel("pairs")
ax.set_title("Time to the follow-up", fontsize=12)
fig.tight_layout(); fig.savefig("charts/3_days_to_followup.png", dpi=200); plt.show()
print(f"Median: {pairs.days_after_merge.median():.1f} days | within 7 days: {(pairs.days_after_merge <= 7).mean():.0%}")

## 4. Result 2 – the list of pairs (AI PR → follow-up PR, shared files) → Excel

In [ ]:
pr = pd.read_parquet(os.path.join(RAW_V5, "pull_request.parquet"), columns=["id", "title", "html_url"]).drop_duplicates("id").set_index("id")
out = pd.DataFrame({
    "ai_agent": pairs.original_agent.map(AGENTS), "ai_pr_url": pairs.original_pr_id.map(pr.html_url),
    "ai_pr_title": pairs.original_pr_id.map(pr.title),
    "followup_agent": pairs.followup_agent.map(AGENTS), "followup_url": pairs.followup_pr_id.map(pr.html_url),
    "followup_title": pairs.followup_pr_id.map(pr.title), "followup_type": pairs.followup_task_type,
    "days_after": pairs.days_after_merge.round(1), "within_30d": pairs.within_30d,
    "shared_files": pairs.shared_files.map(lambda x: "; ".join(x))})
display(out.head(10))
with pd.ExcelWriter("tables/linker_v5_results.xlsx") as xw:
    per_agent.to_excel(xw, sheet_name="1 Per agent", index=False)
    pd.DataFrame({"pairs": types, "%": pct}).rename_axis("type").reset_index().to_excel(xw, sheet_name="2 Follow-up types", index=False)
    who.reset_index().to_excel(xw, sheet_name="3 Same or other agent", index=False)
    out.head(1_000_000).to_excel(xw, sheet_name="4 All pairs", index=False)
print(f"Saved tables/linker_v5_results.xlsx ({len(out):,} pairs) and charts/*.png")
print("The 30-PR sample for the manual check is made by step6_test_30_prs.py")